In [ ]:
import re
from pathlib import Path
from collections import defaultdict
from io import BytesIO

import torch
import timm
import pickle
from PIL import Image
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Config ---
IMAGE_DIR = Path("../data/raw/BarHill2026/Bar Hill 2026")
MODEL_NAME = "hf-hub:BVRA/MegaDescriptor-L-384"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TOP_K = 10
THUMB_SIZE = (200, 200)

# --- Load MegaDescriptor ---
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0).to(DEVICE).eval()
data_cfg = timm.data.resolve_data_config({}, model=model)
transform = timm.data.create_transform(**data_cfg)

# --- Collect files and labels ---
id_pattern = re.compile(r"(GCN\d+)")
filenames = sorted(f.name for f in IMAGE_DIR.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
labels = [id_pattern.search(f).group(1) for f in filenames]

date_pattern = re.compile(r"-(\d{8}-(?:AM|PM))-")

def get_date(fname):
    match = date_pattern.search(fname)
    return match.group(1) if match else "unknown"

dates = [get_date(f) for f in filenames]

# --- Extract embeddings ---
embeddings, thumbnails = [], []
with torch.no_grad():
    for fname in filenames:
        img = Image.open(IMAGE_DIR / fname).convert("RGB")
        x = transform(img).unsqueeze(0).to(DEVICE)
        emb = model(x).squeeze(0).cpu().numpy()
        embeddings.append(emb / np.linalg.norm(emb))  # normalize for cosine sim

        thumb = img.resize(THUMB_SIZE)
        buf = BytesIO()
        thumb.save(buf, format="JPEG", quality=85)
        thumbnails.append(buf.getvalue())

embeddings = np.stack(embeddings)  # (n_images, dim)
labels = np.array(labels)
filenames = np.array(filenames)

# --- Cosine similarity between all image pairs ---
sim_matrix = embeddings @ embeddings.T  # already normalized -> cosine similarity

# --- For each query newt, find top-K most similar *other* newts (best single-image match) ---
unique_ids = sorted(set(labels))

# Save data for streamlit
data = {
    "filenames": filenames,
    "labels": np.array(labels),
    "dates": dates,
    "embeddings": np.stack(embeddings),
    "thumbnails": thumbnails,  # list of JPEG bytes
}

with open("newt_data.pkl", "wb") as f:
    pickle.dump(data, f)

print(f"Saved {len(filenames)} entries to newt_data.pkl")

def top_matches_for_newt(query_id, k=TOP_K):
    query_idx = np.where(labels == query_id)[0]
    other_idx = np.where(labels != query_id)[0]

    # best similarity per other image, only considering query's images
    sub_sim = sim_matrix[np.ix_(query_idx, other_idx)]  # (n_query_imgs, n_other_imgs)
    best_per_other_img = sub_sim.max(axis=0)  # best match across query's images

    # aggregate to best score per other newt
    scores_by_newt = defaultdict(float)
    best_img_by_newt = {}
    for score, idx in zip(best_per_other_img, other_idx):
        newt_id = labels[idx]
        if score > scores_by_newt[newt_id]:
            scores_by_newt[newt_id] = score
            best_img_by_newt[newt_id] = filenames[idx]

    ranked = sorted(scores_by_newt.items(), key=lambda x: -x[1])[:k]
    return [(newt_id, score, best_img_by_newt[newt_id]) for newt_id, score in ranked]

# --- Interactive dropdown viz ---
dropdown_options = [(f"{nid} ({get_date(filenames[np.where(labels == nid)[0][0]])})", nid) for nid in unique_ids]
dropdown = widgets.Dropdown(options=dropdown_options, description="Newt:")
output = widgets.Output()

def on_change(change):
    with output:
        clear_output(wait=True)
        query_id = change["new"]
        query_img = filenames[np.where(labels == query_id)[0][0]]
        print(f"Query: {query_id}  (showing: {query_img})")
        display(Image.open(IMAGE_DIR / query_img).resize((200, 200)))

        print(f"\nTop {TOP_K} most similar newts:")
        matches = top_matches_for_newt(query_id)

        n_cols = 5
        chunks = [matches[i:i + n_cols] for i in range(0, len(matches), n_cols)]

        grid_rows = []
        for chunk in chunks:
            row_items = []
            for newt_id, score, best_img in chunk:
                img = Image.open(IMAGE_DIR / best_img).convert("RGB").resize((150, 150))
                buf = BytesIO()
                img.save(buf, format="PNG")
                img_widget = widgets.Image(value=buf.getvalue(), format="png", layout=widgets.Layout(width="150px", height="150px"))
                widgets.Label(f"{newt_id} ({score:.3f})"),
                widgets.Label(f"{get_date(best_img)}")
                row_items.append(widgets.VBox([img_widget, label_widget]))
            grid_rows.append(widgets.HBox(row_items))

        display(widgets.VBox(grid_rows))

dropdown.observe(on_change, names="value")
display(dropdown, output)
on_change({"new": unique_ids[0]})  # trigger initial render

Saved 191 entries to newt_data.pkl


Dropdown(description='Newt:', options=(('GCN001 (20260420-PM)', np.str_('GCN001')), ('GCN002 (20260420-PM)', n…

Output()